# Lower Sorbian Dataset Analysis
This notebook creates the 3 needed datasets without lemma overlap.

In [4]:
# Step 1: Invert columns
input_file = 'dsb_original'
output_file = 'dsb'
with open(input_file, 'r', encoding='utf8') as fin, open(output_file, 'w', encoding='utf8') as fout:
    for line in fin:
        parts = line.strip().split('\t')
        if len(parts) == 3:
            lemma, form, msd = parts
            fout.write(f'{lemma}\t{msd}\t{form}\n')
print(f'Inverted columns and saved to {output_file}')

Inverted columns and saved to dsb


In [2]:
# Step 2: Deterministic POS-stratified splits using all verbs and preserving global POS ratios
# Assumptions:
#  - We split ALL rows from input_file (no filtering), using POS from MSD prefix:
#      V = verbs, N = nouns, A = adjectives, O = other
#  - Target ratios are 80/10/10 for train/dev/test (edit the constants below as needed)
#  - Deterministic via sorting (lemma, msd, form) and round-robin-like slicing per POS group

from collections import defaultdict
import os

# Configurable ratios
train_ratio, dev_ratio, test_ratio = 0.80, 0.10, 0.10

# Read all rows from the original source and invert to (lemma, msd, form)
all_by_pos = { 'V': [], 'N': [], 'A': [], 'O': [] }

with open(input_file, 'r', encoding='utf8') as fin:
    for raw in fin:
        parts = raw.rstrip('\n').split('\t')
        if len(parts) != 3:
            continue
        lemma, form, msd = parts
        # Normalize to standard order used by other datasets
        rec = (lemma, msd, form)
        # POS bucket by MSD prefix
        if msd.startswith('V'):
            all_by_pos['V'].append(rec)
        elif msd.startswith('N'):
            all_by_pos['N'].append(rec)
        elif msd.startswith('A'):
            all_by_pos['A'].append(rec)
        else:
            all_by_pos['O'].append(rec)

# Deterministic sort within each POS group
for k in list(all_by_pos.keys()):
    all_by_pos[k] = sorted(all_by_pos[k], key=lambda t: (t[0], t[1], t[2]))

# Build splits preserving ratios per POS (use floor for train/dev; remainder to test)
train_lines, dev_lines, test_lines = [], [], []

def split_group(group):
    n = len(group)
    n_train = int(n * train_ratio)
    n_dev = int(n * dev_ratio)
    n_test = n - n_train - n_dev
    return group[:n_train], group[n_train:n_train+n_dev], group[n_train+n_dev:]

for pos in ['V','N','A','O']:
    g = all_by_pos[pos]
    tr, dv, ts = split_group(g)
    train_lines.extend(tr)
    dev_lines.extend(dv)
    test_lines.extend(ts)

# Verify POS ratios per split
from collections import Counter

def pos_counts(lines):
    c = Counter()
    for lemma, msd, form in lines:
        if msd.startswith('V'):
            c['V'] += 1
        elif msd.startswith('N'):
            c['N'] += 1
        elif msd.startswith('A'):
            c['A'] += 1
        else:
            c['O'] += 1
    return c

c_all = { k: len(all_by_pos[k]) for k in ['V','N','A','O'] }
c_tr, c_dv, c_ts = pos_counts(train_lines), pos_counts(dev_lines), pos_counts(test_lines)

print("Totals by POS:", c_all)
print("Train POS:", c_tr)
print("Dev   POS:", c_dv)
print("Test  POS:", c_ts)
print("Sizes (train/dev/test):", len(train_lines), len(dev_lines), len(test_lines))

# Write files next to output_file, using a different base to avoid clobbering earlier artifacts
base = 'dsb_all'
trn_path = f"{base}.trn"
dev_path = f"{base}.dev"
tst_path = f"{base}.tst"

with open(trn_path, 'w', encoding='utf8') as ftr:
    for lemma, msd, form in train_lines:
        ftr.write(f"{lemma}\t{msd}\t{form}\n")
with open(dev_path, 'w', encoding='utf8') as fdev:
    for lemma, msd, form in dev_lines:
        fdev.write(f"{lemma}\t{msd}\t{form}\n")
with open(tst_path, 'w', encoding='utf8') as ftst:
    for lemma, msd, form in test_lines:
        ftst.write(f"{lemma}\t{msd}\t{form}\n")

print("Wrote:")
print(" -", trn_path, len(train_lines))
print(" -", dev_path, len(dev_lines))
print(" -", tst_path, len(test_lines))

# Guarantee: all verbs are used (they are split among the three); verify no verb loss
verb_total = c_all['V']
verb_used = c_tr['V'] + c_dv['V'] + c_ts['V']
print(f"Verb usage check: total={verb_total}, used={verb_used}, ok={verb_total == verb_used}")


Totals by POS: {'V': 2898, 'N': 12372, 'A': 4851, 'O': 0}
Train POS: Counter({'N': 9897, 'A': 3880, 'V': 2318})
Dev   POS: Counter({'N': 1237, 'A': 485, 'V': 289})
Test  POS: Counter({'N': 1238, 'A': 486, 'V': 291})
Sizes (train/dev/test): 16095 2011 2015
Wrote:
 - dsb_all.trn 16095
 - dsb_all.dev 2011
 - dsb_all.tst 2015
Verb usage check: total=2898, used=2898, ok=True


# Step 3: Create fixed-size splits (10k/1k/1k) with minimal lemma overlap
Goal: deterministically build train/dev/test with sizes 10,000 / 1,000 / 1,000 from the verbs-only file `dsb` (created in Step 1), avoiding lemma overlap as much as possible. If exact sizes cannot be met without overlap, we only split lemmas across splits as a last resort to hit the exact counts.

In [5]:
# Step 3: Deterministic 10k/1k/1k splits with minimal lemma overlap (verbs-only)
import os
from collections import defaultdict

# Inputs/outputs
input_path = 'dsb'  # verbs-only, columns: lemma\tmsd\tform (written by Step 1)
base_out = 'dsb'  # will write base_out.trn/dev/tst

# Targets
TRAIN_TARGET, DEV_TARGET, TEST_TARGET = 10_000, 1_000, 1_000

if not os.path.exists(input_path):
    print(f"Input file '{input_path}' not found. Run Step 1 or adjust 'input_path'.")
else:
    # Read and group by lemma
    lemma_groups = defaultdict(list)
    total_rows = 0
    with open(input_path, 'r', encoding='utf8') as fin:
        for raw in fin:
            parts = raw.rstrip('\n').split('\t')
            if len(parts) != 3:
                continue
            lemma, msd, form = parts  # dsb is already (lemma, msd, form)
            lemma_groups[lemma].append((lemma, msd, form))
            total_rows += 1

    desired_total = TRAIN_TARGET + DEV_TARGET + TEST_TARGET
    if total_rows < desired_total:
        print(f"Warning: dataset has only {total_rows} rows < desired {desired_total}. Targets will be reduced.")
        # Keep dev/test up to requested, reduce train first
        train_target = max(0, min(TRAIN_TARGET, total_rows - DEV_TARGET - TEST_TARGET))
        rem = total_rows - train_target
        dev_target = min(DEV_TARGET, rem // 2)
        test_target = min(TEST_TARGET, rem - dev_target)
    else:
        train_target, dev_target, test_target = TRAIN_TARGET, DEV_TARGET, TEST_TARGET

    # Deterministic ordering: pack larger lemmas first to minimize future splitting
    # Sort by (group size desc, lemma asc) and sort examples within lemma for determinism
    for lem in list(lemma_groups.keys()):
        lemma_groups[lem] = sorted(lemma_groups[lem], key=lambda t: (t[1], t[2]))
    ordered_lemmas = sorted(lemma_groups.keys(), key=lambda L: (-len(lemma_groups[L]), L))

    splits = { 'train': [], 'dev': [], 'test': [] }
    targets = { 'train': train_target, 'dev': dev_target, 'test': test_target }
    remaining = targets.copy()

    # First pass: assign whole lemmas if they fit entirely in a split's remaining budget
    pending = []  # lemmas too large to fit in any split's remaining budget at time of consideration
    for lem in ordered_lemmas:
        group = lemma_groups[lem]
        gsize = len(group)
        # candidate splits that can fully accommodate now
        candidates = [s for s in ['train','dev','test'] if remaining[s] >= gsize]
        if candidates:
            # choose the split with the largest remaining room (stable priority: train > dev > test on ties)
            best = max(candidates, key=lambda s: (remaining[s], 1 if s=='train' else 0 if s=='dev' else -1))
            splits[best].extend(group)
            remaining[best] -= gsize
        else:
            pending.append(lem)

    # Second pass: fill deficits by partially assigning remaining lemmas (introduces minimal overlap)
    overlap_lemmas = set()
    if any(remaining.values()) and pending:
        # Iterate pending lemmas in deterministic order (same as before)
        for lem in pending:
            grp = lemma_groups[lem]
            idx = 0
            for _ in range(3):  # at most iterate 3 splits per lemma
                if sum(remaining.values()) == 0 or idx >= len(grp):
                    break
                # pick split with largest remaining
                best = max(['train','dev','test'], key=lambda s: remaining[s])
                if remaining[best] <= 0:
                    break
                take = min(remaining[best], len(grp) - idx)
                if take > 0:
                    splits[best].extend(grp[idx:idx+take])
                    remaining[best] -= take
                    idx += take
                    overlap_lemmas.add(lem)

    # Sanity trims (should be no-ops, but keep deterministic trimming if any overshoot occurred)
    for s, tgt in targets.items():
        if len(splits[s]) > tgt:
            splits[s] = splits[s][:tgt]

    # Report sizes and lemma overlap stats
    def lemma_set(lines):
        return {lem for (lem, _msd, _form) in lines}
    lem_train, lem_dev, lem_test = map(lemma_set, (splits['train'], splits['dev'], splits['test']))
    inter_tr_dev = len(lem_train & lem_dev)
    inter_tr_ts  = len(lem_train & lem_test)
    inter_dev_ts = len(lem_dev & lem_test)

    print("Targets:", targets)
    print("Final sizes:", {k: len(v) for k,v in splits.items()})
    print("Unique lemmas per split:", {'train': len(lem_train), 'dev': len(lem_dev), 'test': len(lem_test)})
    print("Lemma overlaps (train∩dev, train∩test, dev∩test):", (inter_tr_dev, inter_tr_ts, inter_dev_ts))
    if overlap_lemmas:
        print(f"Lemmas forcibly split across splits to reach exact sizes: {len(overlap_lemmas)}")

    # Write outputs
    trn_path = f"{base_out}.trn"
    dev_path = f"{base_out}.dev"
    tst_path = f"{base_out}.tst"

    with open(trn_path, 'w', encoding='utf8') as ftr:
        for lemma, msd, form in splits['train']:
            ftr.write(f"{lemma}\t{msd}\t{form}\n")
    with open(dev_path, 'w', encoding='utf8') as fdev:
        for lemma, msd, form in splits['dev']:
            fdev.write(f"{lemma}\t{msd}\t{form}\n")
    with open(tst_path, 'w', encoding='utf8') as ftst:
        for lemma, msd, form in splits['test']:
            ftst.write(f"{lemma}\t{msd}\t{form}\n")

    print("Wrote:")
    print(" -", trn_path, len(splits['train']))
    print(" -", dev_path, len(splits['dev']))
    print(" -", tst_path, len(splits['test']))

Targets: {'train': 10000, 'dev': 1000, 'test': 1000}
Final sizes: {'train': 10000, 'dev': 1000, 'test': 1000}
Unique lemmas per split: {'train': 411, 'dev': 57, 'test': 57}
Lemma overlaps (train∩dev, train∩test, dev∩test): (1, 1, 1)
Lemmas forcibly split across splits to reach exact sizes: 1
Wrote:
 - dsb.trn 10000
 - dsb.dev 1000
 - dsb.tst 1000
